In [ ]:
# Lab type: review
# Course: DS202 — Time Series Analysis & Forecasting
# Lesson: What Makes Time Series Different
# Task: The code below is correct and runs as intended. Your job is judgment,
#       not debugging: run each section, then answer the judgment questions
#       in the markdown cells. Write 2-4 sentences per answer.

# Lab: Two Evaluations of the Same Model

This lab shows you the same model evaluated two ways — a shuffled split and a
chronological split — on the course's daily orders series. **Both code blocks are
correct implementations of what they claim to do.** The question is which claims
matter, and when.

**Outputs are cleared.** Run each cell to generate results.

## Setup

In [ ]:
!pip install pandas numpy scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
days = pd.date_range("2023-01-01", "2025-12-31", freq="D")
t = np.arange(len(days))

trend   = 200 + 0.15 * t
weekday = np.array([-14, -18, -11, -6, 9, 52, 61])[days.dayofweek]
yearly  = 38 * np.sin(2 * np.pi * (days.dayofyear - 320) / 365.25)
noise   = rng.normal(0, 16, len(days))

orders = pd.Series(trend + weekday + yearly + noise, index=days, name="orders").round()
print(f"{len(orders)} days, {orders.index[0].date()} to {orders.index[-1].date()}")
orders.head()

## Section 1: The structure in the ordering

Autocorrelation before and after shuffling — the Lesson 1 experiment, reproduced.

In [ ]:
print(f"lag-1 autocorrelation: {orders.autocorr(lag=1):.3f}")
print(f"lag-7 autocorrelation: {orders.autocorr(lag=7):.3f}")

shuffled = pd.Series(rng.permutation(orders.to_numpy()), index=orders.index)
print(f"lag-1 after shuffling:  {shuffled.autocorr(lag=1):.3f}")

**Question 1.** The shuffled series contains the identical 1,096 values, yet its
autocorrelation is ≈ 0. A colleague argues: "the information is still there — same numbers,
same mean, same variance." What information, exactly, was destroyed? Name one business
question you could still answer with the shuffled data, and one you no longer can.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 1</summary>

What was destroyed is the *conditional* structure: which value follows which. The
distribution (mean, variance, histogram) survives shuffling, so questions about the
overall level — "what's a typical day's order count?", "what fraction of days exceed
350 orders?" — are still answerable. What's gone is everything sequential: trend
("are we growing?"), seasonality ("which weekday peaks?"), and any forecast
("what happens tomorrow?"). Forecasting is precisely the exploitation of order,
which is why autocorrelation collapsing to zero means forecastability collapsed too.

</details>

## Section 2: Two evaluations, one model

Both evaluations below are *correctly implemented*. Read them carefully before running.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

df = pd.DataFrame({"orders": orders})
df["dayofweek"] = df.index.dayofweek
df["lag_1"] = df["orders"].shift(1)
df["lag_7"] = df["orders"].shift(7)
df = df.dropna()

X, y = df[["dayofweek", "lag_1", "lag_7"]], df["orders"]

# Evaluation A: shuffled 80/20 split
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X, y, test_size=0.2, random_state=0)
model_a = HistGradientBoostingRegressor(random_state=0).fit(Xa_tr, ya_tr)
print(f"Evaluation A (shuffled):      MAE {mean_absolute_error(ya_te, model_a.predict(Xa_te)):.1f}")

# Evaluation B: chronological split — last 20% of days as the test block
cut = int(len(df) * 0.8)
Xb_tr, Xb_te, yb_tr, yb_te = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]
model_b = HistGradientBoostingRegressor(random_state=0).fit(Xb_tr, yb_tr)
print(f"Evaluation B (chronological): MAE {mean_absolute_error(yb_te, model_b.predict(Xb_te)):.1f}")

**Question 2.** Evaluation A reports a lower (better) MAE than Evaluation B. Explain
*mechanically* where A's advantage comes from — what does model A get to see during
training that model B doesn't, and why does that matter for the number reported?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 2</summary>

In the shuffled split, the test days are scattered through the full three years, so for
almost every test day the model trained on its immediate neighbours — often the day
before and the day after. With lag features and a trending series, that means model A
was trained on the level of the series *around every test point*, including levels from
after it. Model B's test block is entirely in the future of its training data, so it must
extrapolate the trend beyond anything it saw — the actual deployed task. A's number
measures interpolation between known neighbours; B's measures forecasting.

</details>

**Question 3.** Is there *any* legitimate use of a shuffled split on temporal data?
Consider: an imputation model that fills historical gaps (like the 9-day outage in
Lesson 2), versus a model that forecasts next week for inventory planning. Justify
which evaluation each task deserves.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 3</summary>

Yes — when the deployed task itself has access to both sides of the gap. A model that
retrospectively fills historical missing values genuinely gets to condition on data
before *and after* the gap, so evaluating it on randomly held-out historical days is
faithful to deployment. The forecasting task never has the future side, so its
evaluation must be chronological. The rule isn't "never shuffle" — it's "the evaluation
must reproduce the information available at deployment time." For forecasting, that
means chronological, every time.

</details>

## Summary

> **Complete each sentence in one line.**

1. Shuffling a time series preserves its ________ but destroys its ________.
2. A shuffled evaluation of a forecasting model measures ________, not forecasting.
3. An evaluation design is valid when it reproduces ________.

<details>
<summary>🔑 Reveal summary answers</summary>

1. Shuffling preserves its **distribution (values, mean, variance)** but destroys its
   **ordering — trend, seasonality, and all autocorrelation**.
2. A shuffled evaluation measures **interpolation between temporally adjacent known
   values**, not forecasting.
3. An evaluation design is valid when it reproduces **the information actually
   available at deployment (prediction) time**.

</details>